# Setup

In [3]:
from pathlib import Path
from dataclasses import dataclass, asdict
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
from torchvision import models
from torchvision.models import ConvNeXt_Tiny_Weights, EfficientNet_B3_Weights, ResNeXt50_32X4D_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [4]:
@dataclass
class CFG:
    seed: int = 42
    model_name: str = "efficientnet_b3"  # convnext_tiny | efficientnet_b3 | resnext50_32x4d
    img_size: int = 320
    batch_size: int = 6
    num_workers: int = 0
    num_workers_test: int = 0
    num_workers_val: int = 0
    epochs: int = 12
    lr: float = 1.2e-4
    weight_decay: float = 1e-4
    patience: int = 3
    n_splits: int = 5
    run_full_cv: bool = True
    mixed_precision: bool = True

    # Loss config
    loss_name: str = "focal"  # ce | ce_ls | focal
    label_smoothing: float = 0.08
    focal_gamma: float = 2.0
    use_class_weights: bool = True

    # TTA config (for test inference)
    use_tta: bool = True
    tta_hflip: bool = True
    tta_light_bc: bool = False
    tta_bc_contrast: float = 1.05
    tta_bc_brightness: float = 0.02

cfg = CFG()
seed_everything(cfg.seed)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_PATH = PROJECT_ROOT / "data" / "splits" / "train_5fold_stratified_group_seed42.csv"
TEST_META_PATH = PROJECT_ROOT / "data" / "processed" / "clean" / "test_metadata_with_hash.csv"
SAMPLE_SUB_PATH = PROJECT_ROOT / "data" / "raw" / "samplesubmission.csv"

CHECKPOINT_DIR = PROJECT_ROOT / "experiments" / "checkpoints" / "baseline"
OOF_DIR = PROJECT_ROOT / "experiments" / "oof_predictions"
SUB_DIR = PROJECT_ROOT / "output" / "submissions"
TEST_PROB_DIR = PROJECT_ROOT / "output" / "test_probabilities"
for p in [CHECKPOINT_DIR, OOF_DIR, SUB_DIR, TEST_PROB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOLDS_TO_RUN = list(range(cfg.n_splits)) if cfg.run_full_cv else [0]

print(f"DEVICE: {DEVICE}")
print(f"FOLDS_TO_RUN: {FOLDS_TO_RUN}")
print(f"Using split file: {SPLIT_PATH}")
print(f"num_workers(train/val/test): {cfg.num_workers}/{cfg.num_workers_val}/{cfg.num_workers_test}")
print(f"Loss setup: {cfg.loss_name} | label_smoothing={cfg.label_smoothing} | focal_gamma={cfg.focal_gamma}")
print(f"TTA setup: use_tta={cfg.use_tta}, hflip={cfg.tta_hflip}, light_bc={cfg.tta_light_bc}")

DEVICE: cuda
FOLDS_TO_RUN: [0, 1, 2, 3, 4]
Using split file: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\data\splits\train_5fold_stratified_group_seed42.csv
num_workers(train/val/test): 0/0/0
Loss setup: focal | label_smoothing=0.08 | focal_gamma=2.0
TTA setup: use_tta=True, hflip=True, light_bc=False


# Data

In [5]:
if not SPLIT_PATH.exists():
    raise FileNotFoundError("Split file tidak ditemukan. Jalankan notebook 02_data_split_prep.ipynb dulu.")
if not TEST_META_PATH.exists():
    raise FileNotFoundError("test_metadata_with_hash.csv tidak ditemukan. Jalankan notebook 02_data_split_prep.ipynb dulu.")

split_df = pd.read_csv(SPLIT_PATH).reset_index(drop=True)
split_df["row_id"] = np.arange(len(split_df))
test_df = pd.read_csv(TEST_META_PATH).reset_index(drop=True)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

labels = sorted(split_df["label"].unique().tolist())
label2idx = {label: i for i, label in enumerate(labels)}
idx2label = {i: label for label, i in label2idx.items()}
split_df["label_idx"] = split_df["label"].map(label2idx).astype(int)

print(f"Train rows in split: {len(split_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Num classes: {len(labels)}")
display(split_df.head())

Train rows in split: 1342
Test rows: 404
Num classes: 6


,label,file_name,file_path,ext,file_size,md5,fold,row_id,label_idx
0,fake_printed,printed_113.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,503348,0009e30b22fee6b9032466fd61cc3658,2,0,2
1,fake_mannequin,mannequin_070.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,84382,0010225da4adf0fc21f41b1274baad5c,4,1,0
2,realperson,real_143.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,551953,004c025fa0e94f1bfeb4c105e844e591,3,2,5
3,fake_mannequin,mannequin_216.jpg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpg,89012,0074ac79a98d1ecd4897e5958b44c281,0,3,0
4,fake_unknown,unknown_206.jpeg,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,.jpeg,29190,0074bbd6358f3ae0985af0fb26651361,0,4,4


# Dataset And Model

In [6]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ColorJitter(p=0.3),
    A.ImageCompression(quality_range=(70, 100), p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

valid_tfms = A.Compose([
    A.Resize(cfg.img_size, cfg.img_size),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class FaceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["file_path"]).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        if self.is_test:
            return img, row["file_name"]
        return img, int(row["label_idx"]), int(row["row_id"])

def build_model(model_name: str, num_classes: int):
    model_name = model_name.lower()
    if model_name == "convnext_tiny":
        model = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
        return model
    if model_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model
    if model_name == "resnext50_32x4d":
        model = models.resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V2)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model
    raise ValueError(f"Unsupported model_name: {model_name}")

class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights=None, gamma: float = 2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.class_weights,
            reduction="none",
        )
        pt = torch.exp(-ce_loss)
        focal = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal.mean()

def build_criterion(class_weights: torch.Tensor | None):
    if not cfg.use_class_weights:
        class_weights = None
    if cfg.loss_name == "ce":
        return nn.CrossEntropyLoss(weight=class_weights)
    if cfg.loss_name == "ce_ls":
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)
    if cfg.loss_name == "focal":
        return WeightedFocalLoss(class_weights=class_weights, gamma=cfg.focal_gamma)
    raise ValueError(f"Unsupported loss_name: {cfg.loss_name}")

def make_loader(df, transform, is_test=False, shuffle=False):
    ds = FaceDataset(df=df, transform=transform, is_test=is_test)
    if is_test:
        workers = cfg.num_workers_test
    else:
        workers = cfg.num_workers if shuffle else cfg.num_workers_val

    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

class_counts = split_df["label"].value_counts()
class_weight_values = []
for label in labels:
    c = class_counts[label]
    class_weight_values.append(len(split_df) / (len(labels) * c))
class_weights = torch.tensor(class_weight_values, dtype=torch.float32)
print("Class weights:", dict(zip(labels, [round(v, 4) for v in class_weight_values])))

Class weights: {'fake_mannequin': np.float64(1.2426), 'fake_mask': np.float64(0.9242), 'fake_printed': np.float64(2.9822), 'fake_screen': np.float64(1.1897), 'fake_unknown': np.float64(0.8223), 'realperson': np.float64(0.581)}


# Train Eval Utils

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0

    pbar = tqdm(loader, total=len(loader), desc="train", leave=False)
    for images, targets, _ in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    probs_all = []
    y_all = []
    row_id_all = []

    pbar = tqdm(loader, total=len(loader), desc="valid", leave=False)
    for images, targets, row_ids in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        probs_all.append(probs)
        y_all.append(targets.detach().cpu().numpy())
        row_id_all.append(row_ids.numpy())

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    probs_all = np.concatenate(probs_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    row_id_all = np.concatenate(row_id_all, axis=0)
    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, y_all, probs_all, row_id_all

def _tta_views(images: torch.Tensor):
    views = [images]
    if cfg.tta_hflip:
        views.append(torch.flip(images, dims=[3]))
    if cfg.tta_light_bc:
        bc = images * cfg.tta_bc_contrast + cfg.tta_bc_brightness
        bc = torch.clamp(bc, -3.0, 3.0)
        views.append(bc)
        if cfg.tta_hflip:
            views.append(torch.flip(bc, dims=[3]))
    return views

@torch.no_grad()
def predict_test(model, loader, use_tta: bool = True):
    model.eval()
    probs_all = []
    file_all = []

    pbar = tqdm(loader, total=len(loader), desc="test", leave=False)
    for images, file_names in pbar:
        images = images.to(DEVICE, non_blocking=True)

        if use_tta:
            views = _tta_views(images)
            probs_acc = None
            for view in views:
                with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
                    logits = model(view)
                probs = torch.softmax(logits, dim=1)
                probs_acc = probs if probs_acc is None else (probs_acc + probs)
            probs = (probs_acc / len(views)).detach().cpu().numpy()
        else:
            with autocast(enabled=cfg.mixed_precision and DEVICE.type == "cuda"):
                logits = model(images)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()

        probs_all.append(probs)
        file_all.extend(list(file_names))

    probs_all = np.concatenate(probs_all, axis=0)
    return probs_all, file_all

# Dataloader Sanity Check

In [8]:
sanity_fold = FOLDS_TO_RUN[0]
sanity_trn = split_df[split_df["fold"] != sanity_fold].reset_index(drop=True)
sanity_val = split_df[split_df["fold"] == sanity_fold].reset_index(drop=True)

sanity_trn_loader = make_loader(sanity_trn, transform=train_tfms, is_test=False, shuffle=True)
sanity_val_loader = make_loader(sanity_val, transform=valid_tfms, is_test=False, shuffle=False)
sanity_test_loader = make_loader(test_df, transform=valid_tfms, is_test=True, shuffle=False)

t0 = time.time()
x, y, rid = next(iter(sanity_trn_loader))
t1 = time.time()
print(f"Train batch loaded in {t1 - t0:.2f}s | x={tuple(x.shape)} y={tuple(y.shape)} rid={tuple(rid.shape)}")

t0 = time.time()
xv, yv, ridv = next(iter(sanity_val_loader))
t1 = time.time()
print(f"Val batch loaded in {t1 - t0:.2f}s | x={tuple(xv.shape)} y={tuple(yv.shape)}")

t0 = time.time()
xt, fn = next(iter(sanity_test_loader))
t1 = time.time()
print(f"Test batch loaded in {t1 - t0:.2f}s | x={tuple(xt.shape)} files={len(fn)}")

Train batch loaded in 0.23s | x=(6, 3, 320, 320) y=(6,) rid=(6,)
Val batch loaded in 0.05s | x=(6, 3, 320, 320) y=(6,)
Test batch loaded in 0.08s | x=(6, 3, 320, 320) files=6


# Train

In [9]:
timestamp = time.strftime("%Y%m%d_%H%M%S")
run_name = f"baseline_{cfg.model_name}_img{cfg.img_size}_{timestamp}"
run_ckpt_dir = CHECKPOINT_DIR / run_name
run_ckpt_dir.mkdir(parents=True, exist_ok=True)

oof_probs = np.full((len(split_df), len(labels)), np.nan, dtype=np.float32)
oof_targets = split_df["label_idx"].values.copy()
test_probs_sum = np.zeros((len(test_df), len(labels)), dtype=np.float32)
fold_scores = []

test_loader = make_loader(test_df, transform=valid_tfms, is_test=True, shuffle=False)
scaler = GradScaler(enabled=cfg.mixed_precision and DEVICE.type == "cuda")

for fold in FOLDS_TO_RUN:
    print("=" * 80)
    print(f"Fold {fold}")

    trn_fold = split_df[split_df["fold"] != fold].reset_index(drop=True)
    val_fold = split_df[split_df["fold"] == fold].reset_index(drop=True)

    trn_loader = make_loader(trn_fold, transform=train_tfms, is_test=False, shuffle=True)
    val_loader = make_loader(val_fold, transform=valid_tfms, is_test=False, shuffle=False)

    model = build_model(cfg.model_name, num_classes=len(labels)).to(DEVICE)
    w = class_weights.to(DEVICE) if cfg.use_class_weights else None
    criterion = build_criterion(w)
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.lr * 0.1)

    best_f1 = -1.0
    best_epoch = -1
    best_path = run_ckpt_dir / f"fold{fold}_best.pt"
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        trn_loss = train_one_epoch(model, trn_loader, criterion, optimizer, scaler)
        val_loss, y_true, val_probs, row_ids = eval_one_epoch(model, val_loader, criterion)
        y_pred = val_probs.argmax(axis=1)
        val_f1 = f1_score(y_true, y_pred, average="macro")

        print(
            f"Epoch {epoch:02d} | trn_loss={trn_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "labels": labels,
                    "label2idx": label2idx,
                    "config": asdict(cfg),
                    "fold": fold,
                    "best_epoch": best_epoch,
                    "best_f1": best_f1,
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping triggered")
                break

        scheduler.step()

    ckpt = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])

    _, y_true, val_probs, row_ids = eval_one_epoch(model, val_loader, criterion)
    y_pred = val_probs.argmax(axis=1)
    val_f1 = f1_score(y_true, y_pred, average="macro")
    fold_scores.append({"fold": fold, "best_epoch": best_epoch, "macro_f1": val_f1})

    oof_probs[row_ids] = val_probs

    test_probs, test_files = predict_test(model, test_loader, use_tta=cfg.use_tta)
    test_probs_sum += test_probs

    print(f"Fold {fold} done | best_epoch={best_epoch} | macro_f1={val_f1:.4f}")

test_probs_avg = test_probs_sum / max(len(FOLDS_TO_RUN), 1)
fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

Fold 0


Epoch 01 | trn_loss=0.8684 | val_loss=0.3884 | val_macro_f1=0.7945


Epoch 02 | trn_loss=0.4238 | val_loss=0.2741 | val_macro_f1=0.8266


Epoch 03 | trn_loss=0.3012 | val_loss=0.2155 | val_macro_f1=0.8672


Epoch 04 | trn_loss=0.2191 | val_loss=0.2043 | val_macro_f1=0.8525


...

Epoch 10 | trn_loss=0.1019 | val_loss=0.1464 | val_macro_f1=0.9048


Epoch 11 | trn_loss=0.0661 | val_loss=0.1469 | val_macro_f1=0.9103


Epoch 12 | trn_loss=0.0728 | val_loss=0.1471 | val_macro_f1=0.9064
Early stopping triggered


Fold 4 done | best_epoch=9 | macro_f1=0.9174


,fold,best_epoch,macro_f1
0,0,11,0.924305
1,1,7,0.903308
2,2,11,0.917352
3,3,3,0.862776
4,4,9,0.917356


# Evaluate OOF

In [8]:
valid_mask = ~np.isnan(oof_probs).any(axis=1)
oof_eval_df = split_df.loc[valid_mask].copy()
oof_eval_probs = oof_probs[valid_mask]
oof_eval_true = oof_eval_df["label_idx"].values
oof_eval_pred = oof_eval_probs.argmax(axis=1)

oof_macro_f1 = f1_score(oof_eval_true, oof_eval_pred, average="macro")
pr, rc, f1c, sup = precision_recall_fscore_support(
    oof_eval_true,
    oof_eval_pred,
    labels=list(range(len(labels))),
    zero_division=0,
    average=None,
)
metrics_df = pd.DataFrame({
    "label": [idx2label[i] for i in range(len(labels))],
    "precision": pr,
    "recall": rc,
    "f1": f1c,
    "support": sup,
})

print(f"OOF evaluated rows: {len(oof_eval_df)} / {len(split_df)}")
print(f"OOF Macro F1: {oof_macro_f1:.5f}")
display(metrics_df)

cm = confusion_matrix(oof_eval_true, oof_eval_pred, labels=list(range(len(labels))))
cm_df = pd.DataFrame(cm, index=[idx2label[i] for i in range(len(labels))], columns=[idx2label[i] for i in range(len(labels))])
display(cm_df)

oof_prob_cols = [f"prob_{idx2label[i]}" for i in range(len(labels))]
oof_save_df = oof_eval_df[["row_id", "file_path", "label", "fold"]].copy()
oof_save_df["pred_label"] = [idx2label[i] for i in oof_eval_pred]
oof_save_df[oof_prob_cols] = oof_eval_probs
oof_save_path = OOF_DIR / f"oof_{run_name}.csv"
oof_save_df.to_csv(oof_save_path, index=False)
print(f"Saved OOF predictions: {oof_save_path}")

OOF evaluated rows: 1342 / 1342
OOF Macro F1: 0.92144


,label,precision,recall,f1,support
0,fake_mannequin,0.943182,0.922222,0.932584,180
1,fake_mask,0.906883,0.925620,0.916155,242
2,fake_printed,0.819277,0.906667,0.860759,75
3,fake_screen,0.967391,0.946809,0.956989,188
4,fake_unknown,0.916968,0.933824,0.925319,272
5,realperson,0.949333,0.924675,0.936842,385


,fake_mannequin,fake_mask,fake_printed,fake_screen,fake_unknown,realperson
fake_mannequin,166,3,1,0,7,3
fake_mask,3,224,4,0,4,7
fake_printed,0,2,68,0,4,1
fake_screen,2,0,2,178,2,4
fake_unknown,5,3,4,2,254,4
realperson,0,15,4,4,6,356


Saved OOF predictions: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\experiments\oof_predictions\oof_baseline_convnext_tiny_img288_20260331_232502.csv


# Submission

In [9]:
test_pred_idx = test_probs_avg.argmax(axis=1)
test_pred_label = [idx2label[i] for i in test_pred_idx]

test_pred_df = pd.DataFrame({
    "id": [Path(fn).stem for fn in test_files],
    "label": test_pred_label,
})

submission_df = sample_sub[["id"]].merge(test_pred_df, on="id", how="left")
if submission_df["label"].isna().any():
    missing_n = int(submission_df["label"].isna().sum())
    raise ValueError(f"Ada {missing_n} id test yang belum terprediksi")

sub_path = SUB_DIR / f"submission_{run_name}.csv"
submission_df.to_csv(sub_path, index=False)
print(f"Saved submission: {sub_path}")
display(submission_df.head())

test_prob_cols = [f"prob_{idx2label[i]}" for i in range(len(labels))]
test_prob_df = pd.DataFrame(test_probs_avg, columns=test_prob_cols)
test_prob_df["id"] = [Path(fn).stem for fn in test_files]
test_prob_df = sample_sub[["id"]].merge(test_prob_df, on="id", how="left")
if test_prob_df[test_prob_cols].isna().any().any():
    raise ValueError("Ada probabilitas test yang missing setelah align ke samplesubmission")

test_prob_path = TEST_PROB_DIR / f"test_probs_{run_name}.csv"
test_prob_df.to_csv(test_prob_path, index=False)
print(f"Saved test probabilities: {test_prob_path}")

config_path = run_ckpt_dir / "config_used.json"
pd.Series(asdict(cfg)).to_json(config_path, indent=2)
print(f"Saved config: {config_path}")

Saved submission: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\output\submissions\submission_baseline_convnext_tiny_img288_20260331_232502.csv


,id,label
0,test_001,fake_screen
1,test_002,fake_screen
2,test_003,realperson
3,test_004,realperson
4,test_005,fake_printed


Saved test probabilities: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\output\test_probabilities\test_probs_baseline_convnext_tiny_img288_20260331_232502.csv
Saved config: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\experiments\checkpoints\baseline\baseline_convnext_tiny_img288_20260331_232502\config_used.json


# Ensemble Experiments

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OOF_BASE_DIR = PROJECT_ROOT / "experiments" / "oof_predictions"
TEST_PROB_BASE_DIR = PROJECT_ROOT / "output" / "test_probabilities"
ENS_OOF_DIR = PROJECT_ROOT / "experiments" / "oof_predictions" / "ensemble"
ENS_SUB_DIR = PROJECT_ROOT / "output" / "submissions" / "ensemble"
ENS_REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ENS_OOF_DIR.mkdir(parents=True, exist_ok=True)
ENS_SUB_DIR.mkdir(parents=True, exist_ok=True)
ENS_REPORT_DIR.mkdir(parents=True, exist_ok=True)

def infer_backbone(run_name: str):
    rn = run_name.lower()
    if "resnext50_32x4d" in rn:
        return "resnext50_32x4d"
    if "efficientnet_b3" in rn:
        return "efficientnet_b3"
    if "convnext_tiny" in rn:
        return "convnext_tiny"
    return "unknown"

def quick_oof_macro_f1(oof_path: Path):
    df = pd.read_csv(oof_path)
    prob_cols = sorted([c for c in df.columns if c.startswith("prob_")])
    if len(prob_cols) == 0:
        return np.nan
    y_true = df["label"].values
    label_names = [c.replace("prob_", "") for c in prob_cols]
    pred_idx = df[prob_cols].values.argmax(axis=1)
    y_pred = np.array(label_names)[pred_idx]
    return float(f1_score(y_true, y_pred, average="macro"))

all_oof_files = sorted(OOF_BASE_DIR.glob("oof_baseline_*.csv"))
if len(all_oof_files) == 0:
    raise RuntimeError("Belum ada artifact OOF baseline untuk ensemble.")

candidate_rows = []
for oof_path in all_oof_files:
    run_name = oof_path.stem.replace("oof_", "")
    test_prob_path = TEST_PROB_BASE_DIR / f"test_probs_{run_name}.csv"
    if not test_prob_path.exists():
        continue
    candidate_rows.append({
        "run_name": run_name,
        "backbone": infer_backbone(run_name),
        "oof_macro_f1": quick_oof_macro_f1(oof_path),
    })

if len(candidate_rows) == 0:
    raise RuntimeError(
        "Tidak ada pasangan OOF + test_prob yang valid. Jalankan cell Submission pada run yang ingin di-ensemble dulu."
    )

candidate_df = pd.DataFrame(candidate_rows).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
display(candidate_df)

selected = []
used_backbones = set()

# Prioritas 1: sertakan run ResNeXt jika tersedia (biar diversity backbone masuk)
resnext_rows = candidate_df[candidate_df["backbone"] == "resnext50_32x4d"]
if len(resnext_rows) > 0:
    rn = resnext_rows.iloc[0]["run_name"]
    selected.append(rn)
    used_backbones.add("resnext50_32x4d")

# Prioritas 2: ambil run terbaik dari backbone lain yang belum dipakai
for _, row in candidate_df.iterrows():
    if len(selected) >= 3:
        break
    rn = row["run_name"]
    bb = row["backbone"]
    if rn in selected:
        continue
    if bb in used_backbones:
        continue
    selected.append(rn)
    used_backbones.add(bb)

# Prioritas 3: isi sisa slot berdasarkan skor terbaik global
for _, row in candidate_df.iterrows():
    if len(selected) >= 3:
        break
    rn = row["run_name"]
    if rn in selected:
        continue
    selected.append(rn)

if len(selected) < 3:
    raise RuntimeError("Butuh minimal 3 run valid untuk ensemble auto-search.")

selected_run_names = selected[:3]
print("Selected runs (by actual run name, diversity-aware):")
for i, rn in enumerate(selected_run_names, start=1):
    print(f"{i}. {rn}")

,run_name,backbone,oof_macro_f1
0,baseline_efficientnet_b3_img288_20260328_213015,efficientnet_b3,0.914914
1,baseline_convnext_tiny_img320_20260328_233846,convnext_tiny,0.912581
2,baseline_resnext50_32x4d_img224_20260328_222704,resnext50_32x4d,0.900436


Selected runs (by actual run name, diversity-aware):
1. baseline_resnext50_32x4d_img224_20260328_222704
2. baseline_efficientnet_b3_img288_20260328_213015
3. baseline_convnext_tiny_img320_20260328_233846


In [3]:
def load_oof(run_name: str):
    p = OOF_BASE_DIR / f"oof_{run_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"OOF file tidak ditemukan: {p}")
    df = pd.read_csv(p)
    prob_cols = sorted([c for c in df.columns if c.startswith("prob_")])
    if len(prob_cols) == 0:
        raise ValueError(f"Tidak ada kolom probabilitas di: {p}")
    df = df.sort_values("row_id").reset_index(drop=True)
    labels_true = df["label"].values
    probs = df[prob_cols].values.astype(np.float32)
    return df, labels_true, probs, prob_cols

def load_test_probs(run_name: str):
    p = TEST_PROB_BASE_DIR / f"test_probs_{run_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Test probabilities file tidak ditemukan: {p}")
    df = pd.read_csv(p).sort_values("id").reset_index(drop=True)
    prob_cols = sorted([c for c in df.columns if c.startswith("prob_")])
    if len(prob_cols) == 0:
        raise ValueError(f"Tidak ada kolom probabilitas test di: {p}")
    probs = df[prob_cols].values.astype(np.float32)
    return df, probs, prob_cols

def evaluate_oof_weights(weights, oof_artifacts, label_names, base_y):
    first_run = next(iter(weights.keys()))
    probs_ens = np.zeros_like(oof_artifacts[first_run][2], dtype=np.float32)
    for run_name, w in weights.items():
        probs_ens += float(w) * oof_artifacts[run_name][2]
    pred_idx = probs_ens.argmax(axis=1)
    pred_label = np.array(label_names)[pred_idx]
    oof_f1 = f1_score(base_y, pred_label, average="macro")
    return oof_f1, probs_ens, pred_label

def weight_token(w: float):
    return f"{w:.4f}".replace(".", "p")

def build_blend_name(prefix: str, weights: dict, ordered_runs: list[str]):
    parts = [f"{rn}-{weight_token(weights[rn])}" for rn in ordered_runs]
    return f"{prefix}_{'__'.join(parts)}"

# Load all needed run artifacts using actual run names
oof_artifacts = {}
test_prob_artifacts = {}
for run_name in selected_run_names:
    oof_artifacts[run_name] = load_oof(run_name)
    test_prob_artifacts[run_name] = load_test_probs(run_name)

anchor_run = selected_run_names[-1]
base_prob_cols = oof_artifacts[anchor_run][3]
label_names = [c.replace("prob_", "") for c in base_prob_cols]

# Validate label order consistency across OOF runs
for run_name, (_, _, _, prob_cols) in oof_artifacts.items():
    if prob_cols != base_prob_cols:
        raise ValueError(f"Urutan kolom probabilitas OOF beda di run: {run_name}")

# Validate same OOF target ordering across runs
base_y = oof_artifacts[anchor_run][1]
for run_name, (_, y, _, _) in oof_artifacts.items():
    if not np.array_equal(base_y, y):
        raise ValueError(f"OOF target order beda di run: {run_name}")

# Validate test probability column alignment
for run_name, (_, _, prob_cols) in test_prob_artifacts.items():
    if prob_cols != base_prob_cols:
        raise ValueError(f"Urutan kolom probabilitas test beda di run: {run_name}")

# Validate submission id alignment on test probability files
base_ids = test_prob_artifacts[anchor_run][0]["id"].tolist()
for run_name, (df_prob, _, _) in test_prob_artifacts.items():
    if df_prob["id"].tolist() != base_ids:
        raise ValueError(f"Urutan id test probability beda di run: {run_name}")

# Ensemble mode: probability-level blend (no hard voting across model labels)
print("Ensemble mode: probability-level blend")

# Automated weight search only
grid_step = 0.05
grid_vals = np.round(np.arange(0.0, 1.0 + grid_step, grid_step), 2)
weight_candidates = []

if len(selected_run_names) != 3:
    raise ValueError("Saat ini weight search dikunci untuk tepat 3 run terpilih.")

r1, r2, r3 = selected_run_names

# Grid search on 3 runs with exact sum = 1
for w1 in grid_vals:
    for w2 in grid_vals:
        w3 = round(1.0 - w1 - w2, 2)
        if w3 < 0.0 or w3 > 1.0:
            continue
        weight_candidates.append({r1: float(w1), r2: float(w2), r3: float(w3)})

# Random search for additional exploration
rng = np.random.default_rng(42)
for _ in range(400):
    vec = rng.dirichlet(alpha=np.array([1.0, 1.0, 1.0]))
    vec = vec / vec.sum()
    weight_candidates.append({r1: float(vec[0]), r2: float(vec[1]), r3: float(vec[2])})

# Deduplicate candidates by rounded signature
uniq = {}
for w in weight_candidates:
    ws = sum(w.values())
    if ws == 0:
        continue
    w_norm = {k: v / ws for k, v in w.items()}
    key = tuple(round(w_norm[rn], 4) for rn in selected_run_names)
    uniq[key] = w_norm

search_rows = []
for _, w in uniq.items():
    oof_f1, _, _ = evaluate_oof_weights(w, oof_artifacts, label_names, base_y)
    w_rounded = {rn: round(w[rn], 4) for rn in selected_run_names}
    blend_name = build_blend_name("autoblend", w_rounded, selected_run_names)
    search_rows.append({
        "ensemble": blend_name,
        "weights": str(w_rounded),
        "oof_macro_f1": oof_f1,
        "run_1": r1,
        "run_2": r2,
        "run_3": r3,
        "w_run_1": w[r1],
        "w_run_2": w[r2],
        "w_run_3": w[r3],
    })

search_df = pd.DataFrame(search_rows).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
top10_auto_df = search_df.head(10).copy()
print("Top-10 auto weight candidates by OOF:")
display(top10_auto_df[["ensemble", "oof_macro_f1", "weights"]])

timestamp = time.strftime("%Y%m%d_%H%M%S")
top10_report_path = ENS_REPORT_DIR / f"ensemble_top10_oof_{timestamp}.csv"
top10_auto_df.to_csv(top10_report_path, index=False)
print(f"Saved top-10 report: {top10_report_path}")

# Generate probability-level ensemble submissions for top-K
top_k_submit = 3
submit_rows = []
for i, row in top10_auto_df.head(top_k_submit).iterrows():
    w = {
        r1: float(row["w_run_1"]),
        r2: float(row["w_run_2"]),
        r3: float(row["w_run_3"]),
    }
    ws = sum(w.values())
    w = {k: v / ws for k, v in w.items()}

    # OOF blended probs (probability average then argmax)
    oof_f1, oof_probs_blend, oof_pred_label = evaluate_oof_weights(w, oof_artifacts, label_names, base_y)
    oof_save_df = oof_artifacts[anchor_run][0][["row_id", "file_path", "label", "fold"]].copy()
    oof_save_df["pred_label"] = oof_pred_label
    oof_save_df[base_prob_cols] = oof_probs_blend

    blend_name = build_blend_name("autoblend", {k: round(v, 4) for k, v in w.items()}, selected_run_names)
    oof_name = f"oof_{blend_name}_{timestamp}.csv"
    oof_save_path = ENS_OOF_DIR / oof_name
    oof_save_df.to_csv(oof_save_path, index=False)

    # Test blended probs (probability average then argmax)
    test_probs_blend = np.zeros_like(test_prob_artifacts[anchor_run][1], dtype=np.float32)
    for run_name, wk in w.items():
        test_probs_blend += wk * test_prob_artifacts[run_name][1]

    test_pred_idx = test_probs_blend.argmax(axis=1)
    test_pred_label = np.array(label_names)[test_pred_idx]
    sub_out_df = pd.DataFrame({"id": base_ids, "label": test_pred_label})
    sub_out_path = ENS_SUB_DIR / f"submission_{blend_name}_{timestamp}.csv"
    sub_out_df.to_csv(sub_out_path, index=False)

    submit_rows.append({
        "rank": i + 1,
        "ensemble_name": blend_name,
        "oof_macro_f1": oof_f1,
        "weights": str({k: round(v, 4) for k, v in w.items()}),
        "oof_file": str(oof_save_path),
        "submission_file": str(sub_out_path),
    })

submit_plan_df = pd.DataFrame(submit_rows).sort_values("oof_macro_f1", ascending=False).reset_index(drop=True)
print("Top submission candidates (probability-level blend, named runs):")
display(submit_plan_df)

submit_plan_path = ENS_REPORT_DIR / f"ensemble_submit_plan_{timestamp}.csv"
submit_plan_df.to_csv(submit_plan_path, index=False)
print(f"Saved submit plan: {submit_plan_path}")

Ensemble mode: probability-level blend
Top-10 auto weight candidates by OOF:


,ensemble,oof_macro_f1,weights
0,autoblend_baseline_resnext50_32x4d_img224_2026...,0.921223,{'baseline_resnext50_32x4d_img224_20260328_222...
1,autoblend_baseline_resnext50_32x4d_img224_2026...,0.921223,{'baseline_resnext50_32x4d_img224_20260328_222...
2,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920969,{'baseline_resnext50_32x4d_img224_20260328_222...
3,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920736,{'baseline_resnext50_32x4d_img224_20260328_222...
4,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920728,{'baseline_resnext50_32x4d_img224_20260328_222...
5,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920728,{'baseline_resnext50_32x4d_img224_20260328_222...
6,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920728,{'baseline_resnext50_32x4d_img224_20260328_222...
7,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920728,{'baseline_resnext50_32x4d_img224_20260328_222...
8,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920728,{'baseline_resnext50_32x4d_img224_20260328_222...
9,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920642,{'baseline_resnext50_32x4d_img224_20260328_222...


Saved top-10 report: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\reports\tables\ensemble_top10_oof_20260329_011951.csv
Top submission candidates (probability-level blend, named runs):


,rank,ensemble_name,oof_macro_f1,weights,oof_file,submission_file
0,1,autoblend_baseline_resnext50_32x4d_img224_2026...,0.921223,{'baseline_resnext50_32x4d_img224_20260328_222...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...
1,2,autoblend_baseline_resnext50_32x4d_img224_2026...,0.921223,{'baseline_resnext50_32x4d_img224_20260328_222...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...
2,3,autoblend_baseline_resnext50_32x4d_img224_2026...,0.920969,{'baseline_resnext50_32x4d_img224_20260328_222...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...,c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-20...


Saved submit plan: c:\Users\Axioo\Documents\Fahmi\ai\ml\findit-2026\reports\tables\ensemble_submit_plan_20260329_011951.csv
